# AI/ML Task 4 – Classification Models, Evaluation Metrics & Handling Imbalanced Data
**Problem:** Binary Classification — Breast Cancer Diagnosis (Malignant vs Benign)  
**Dataset:** scikit-learn built-in Breast Cancer dataset  
**Author:** Maincrafts Technology Intern  
**Tools:** Python · pandas · NumPy · scikit-learn · matplotlib

In [ ]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print("All libraries imported successfully.")

In [ ]:
# Step 2: Load Dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Target classes: 0 = Malignant, 1 = Benign
print("Dataset shape:", X.shape)
print("Class distribution:", np.bincount(y), "-> [Malignant, Benign]")
print(f"Malignant: {np.bincount(y)[0]} ({100*np.bincount(y)[0]/len(y):.1f}%)")
print(f"Benign:    {np.bincount(y)[1]} ({100*np.bincount(y)[1]/len(y):.1f}%)")
X.head()

In [ ]:
# Step 3: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print()
print("Stratification preserves the original class distribution in both train and test sets,")
print("which matters a lot for imbalanced data - a random split could otherwise under-represent")
print("the minority class in the test set, giving a misleading evaluation.")

In [ ]:
# Step 4: Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Scaling applied. Logistic Regression is distance/gradient-sensitive, so this matters.")

In [ ]:
# Step 5: Train Baseline Classification Model - Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print("Baseline Logistic Regression trained.")

In [ ]:
# Step 6: Confusion Matrix & Classification Report
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print()
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (TN)  = {tn}  -> Correctly predicted Malignant")
print(f"False Positives (FP) = {fp}  -> Predicted Benign, actually Malignant  (DANGEROUS)")
print(f"False Negatives (FN) = {fn}  -> Predicted Malignant, actually Benign (less risky)")
print(f"True Positives (TP)  = {tp}  -> Correctly predicted Benign")

### Why accuracy alone is insufficient

Accuracy treats every error equally. But in medical diagnosis, **a False Negative
(missing a malignant tumor) is far more dangerous than a False Positive** (flagging
a benign tumor as malignant, which leads to extra testing but not a missed diagnosis).

A model could score 95%+ accuracy by simply predicting the majority class most of
the time - while still missing every single malignant case. This is why **Recall**
on the malignant class, and metrics like **F1-score**, matter more than accuracy
in imbalanced, high-stakes classification problems.

In [ ]:
# Step 7: Precision, Recall & F1-Score (explicit calculation)
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)   # of predicted Benign, how many really are
rec  = recall_score(y_test, y_pred)      # of actual Benign, how many did we catch
f1   = f1_score(y_test, y_pred)          # harmonic mean of precision & recall

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

**Key questions answered:**

- **Which metric matters most for medical diagnosis?** Recall on the malignant
  (minority/critical) class - missing a cancer diagnosis (false negative) is the
  costliest error.
- **What happens if recall is low?** Many true malignant cases get classified as
  benign - patients go undiagnosed and untreated. This is the worst-case failure
  mode for a screening tool.
- **Why is F1-score preferred for imbalanced data?** Accuracy can look great while
  the model ignores the minority class entirely. F1 balances precision and recall
  into one number, so a model can't hide poor minority-class performance behind a
  high overall accuracy.

In [ ]:
# Step 8: ROC Curve & AUC Score
y_prob = model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0,1], [0,1], linestyle="--", color='red', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("roc_curve.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC Score: {auc:.4f}")
print("AUC close to 1.0 means the model separates the two classes almost perfectly.")

In [ ]:
# Step 9: Handle Class Imbalance - Class Weighting
model_balanced = LogisticRegression(class_weight="balanced", max_iter=1000)
model_balanced.fit(X_train_scaled, y_train)
y_pred_bal = model_balanced.predict(X_test_scaled)

acc_bal  = accuracy_score(y_test, y_pred_bal)
prec_bal = precision_score(y_test, y_pred_bal)
rec_bal  = recall_score(y_test, y_pred_bal)
f1_bal   = f1_score(y_test, y_pred_bal)
cm_bal   = confusion_matrix(y_test, y_pred_bal)

print("Balanced Model Confusion Matrix:")
print(cm_bal)
print()
print(f"Accuracy : {acc_bal:.4f}  (baseline: {acc:.4f})")
print(f"Precision: {prec_bal:.4f}  (baseline: {prec:.4f})")
print(f"Recall   : {rec_bal:.4f}  (baseline: {rec:.4f})")
print(f"F1-Score : {f1_bal:.4f}  (baseline: {f1:.4f})")
print()
print("class_weight='balanced' automatically re-weights each class inversely")
print("proportional to its frequency, forcing the model to pay more attention")
print("to the minority class during training.")

In [ ]:
# Step 10: Compare with Another Classifier - Decision Tree
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)   # Trees don't require feature scaling
y_pred_tree = tree.predict(X_test)

acc_tree  = accuracy_score(y_test, y_pred_tree)
prec_tree = precision_score(y_test, y_pred_tree)
rec_tree  = recall_score(y_test, y_pred_tree)
f1_tree   = f1_score(y_test, y_pred_tree)
cm_tree   = confusion_matrix(y_test, y_pred_tree)

train_acc_tree = accuracy_score(y_train, tree.predict(X_train))

print("Decision Tree Confusion Matrix:")
print(cm_tree)
print()
print(classification_report(y_test, y_pred_tree, target_names=['Malignant','Benign']))
print(f"Train Accuracy: {train_acc_tree:.4f}  |  Test Accuracy: {acc_tree:.4f}")
print(f"Gap: {train_acc_tree - acc_tree:.4f}  <- signals some overfitting (unconstrained tree)")

**Logistic Regression vs Decision Tree:**

| Aspect | Logistic Regression | Decision Tree |
|---|---|---|
| Stability | More stable, smooth decision boundary | More sensitive to small data changes |
| Interpretability | Coefficients show feature direction/strength | Easy to visualize as rules, but less smooth |
| Overfitting | Low risk (linear model, regularized) | High risk without max_depth/min_samples constraints |
| Performance here | Higher accuracy & AUC | Slightly lower; train accuracy of 1.0 vs lower test accuracy shows overfitting |

Logistic Regression generalizes better here because the classes are close to
linearly separable in this feature space, and the unconstrained Decision Tree
overfits the training data (100% train accuracy).

In [ ]:
# Visualization: Confusion Matrix, ROC Curve, Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Task 4 - Classification Models, Evaluation Metrics & Class Imbalance",
             fontsize=12, fontweight='bold')

# Plot 1: Confusion Matrix heatmap (baseline model)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Malignant','Benign']); axes[0].set_yticklabels(['Malignant','Benign'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix\n(Logistic Regression)')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black',
                     fontsize=14, fontweight='bold')

# Plot 2: ROC Curve
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1], [0,1], 'r--', lw=1, label='Random guess')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(fontsize=9)

# Plot 3: Model comparison across all metrics
mnames = ['Logistic\nReg', 'Logistic Reg\n(Balanced)', 'Decision\nTree']
metrics_data = {
    'Accuracy':  [acc, acc_bal, acc_tree],
    'Precision': [prec, prec_bal, prec_tree],
    'Recall':    [rec, rec_bal, rec_tree],
    'F1':        [f1, f1_bal, f1_tree],
}
x = np.arange(len(mnames)); w = 0.2
colors_list = ['steelblue','salmon','seagreen','goldenrod']
for i, (mname, vals_list) in enumerate(metrics_data.items()):
    axes[2].bar(x + i*w - 1.5*w, vals_list, w, label=mname, color=colors_list[i])
axes[2].set_xticks(x); axes[2].set_xticklabels(mnames, fontsize=8)
axes[2].set_ylabel('Score'); axes[2].set_title('Model Comparison\n(All Metrics)')
axes[2].legend(fontsize=7); axes[2].set_ylim(0,1.15)

plt.tight_layout()
plt.savefig("task4_plots.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model Comparison Summary Table
results = {
    "Model":     ["Logistic Regression", "Logistic Regression (Balanced)", "Decision Tree"],
    "Accuracy":  [acc, acc_bal, acc_tree],
    "Precision": [prec, prec_bal, prec_tree],
    "Recall":    [rec, rec_bal, rec_tree],
    "F1 Score":  [f1, f1_bal, f1_tree],
    "AUC":       [auc, None, None],
}
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


## Final Model Selection Justification

### Selected Model: Baseline Logistic Regression

| Model | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.9825 | 0.9861 | 0.9861 | 0.9861 | 0.9954 |
| Logistic Regression (Balanced) | 0.9561 | 0.9855 | 0.9444 | 0.9645 | - |
| Decision Tree | 0.9123 | 0.9559 | 0.9028 | 0.9286 | - |

**Metric selection:** Recall on the malignant class and F1-score were prioritized over
raw accuracy, since this is a medical diagnosis problem where missing a malignant
case (false negative) carries the highest cost.

**Imbalance handling:** The dataset has a mild imbalance (37% malignant / 63% benign).
class_weight='balanced' was tested - it traded some precision for being more
sensitive to the minority class, but in this case the baseline model already achieved
very high recall (0.9861) on its own, so the balanced version was not strictly
necessary, though it remains a useful technique for more severely imbalanced datasets.

**Final decision:** The baseline Logistic Regression was selected because it achieved
the highest accuracy, F1-score, and AUC (0.9954) of all three models, with only
1 false positive and 1 false negative out of 114 test cases -
an excellent, well-balanced result that did not require correction via class weighting.

**Trade-offs:** Logistic Regression is more interpretable and stable than the Decision
Tree, which showed signs of overfitting (100% train accuracy vs 0.9123 test accuracy).
